# 02 - Your first agent

Now we call Claude for real. By the end of this notebook you will have run an
agent, watched every message the loop produced, and be able to explain what each
one was.

**Cost:** the cells here total well under $0.10 on a normal model.

**Docs:** [Quickstart](https://code.claude.com/docs/en/agent-sdk/quickstart)

## 2.1 Install and authenticate



In [ ]:
# --- boot cell: run this first in every notebook -------------------------
import os, sys, json, asyncio
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")

assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY is not set. Copy .env.example to .env and add your key."
)

MODEL = os.environ.get("COURSE_MODEL") or None   # None = SDK default
print("key loaded, model =", MODEL or "(SDK default)")

## 2.2 `await`, not `asyncio.run()`

The SDK is async. In a **script** you write `asyncio.run(main())`. In a
**notebook** the kernel already has an event loop running, so `asyncio.run()`
raises `RuntimeError: asyncio.run() cannot be called from a running event loop`.

In notebooks, use top-level `await` directly. Every notebook in this course does.
When you lift this code into a script, wrap it in `async def main()` and call
`asyncio.run(main())`. The code inside is identical.

## 2.3 The smallest possible agent

`query()` is the main entry point. It takes a prompt and options, and returns an
**async iterator of messages**. Iterating it *is* running the agent loop.

`tools=[]` removes every built-in tool, so this first call is a plain text turn
with nothing to go wrong.

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions

async for message in query(
    prompt="In one sentence: what is an AI agent?",
    options=ClaudeAgentOptions(tools=[], max_turns=1, model=MODEL),
):
    print(type(message).__name__, message)

Three message types went past: `SystemMessage`, `AssistantMessage`,
`ResultMessage`. That is the minimum shape of any run.

Now let us actually read them.

In [ ]:
from claude_agent_sdk import (
    AssistantMessage, ResultMessage, SystemMessage, TextBlock,
)

async for message in query(
    prompt="In one sentence: what is an AI agent?",
    options=ClaudeAgentOptions(tools=[], max_turns=1, model=MODEL),
):
    if isinstance(message, SystemMessage):
        print(f"[system:{message.subtype}] session {message.data.get('session_id', '?')[:8]}")

    elif isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock):
                print(f"[assistant] {block.text}")

    elif isinstance(message, ResultMessage):
        print(f"[result] {message.subtype} | {message.num_turns} turn(s) "
              f"| ${message.total_cost_usd or 0:.5f}")

## 2.4 The five message types

| Type | When it arrives | What it carries |
|---|---|---|
| `SystemMessage` | session lifecycle | `subtype="init"` (session metadata), `"compact_boundary"`, `"informational"` |
| `AssistantMessage` | after every Claude response | `content`: a list of blocks (text, tool calls, thinking) |
| `UserMessage` | after every tool execution | the tool result being fed back to Claude |
| `StreamEvent` | only with partial messages on | raw token deltas — see notebook 06 |
| `ResultMessage` | once, at the end | final text, cost, usage, turns, session id |

In Python you check with `isinstance()`. In TypeScript you check the `type`
string field.

**Content blocks** inside `AssistantMessage.content`:

- `TextBlock` — `.text`, Claude's prose
- `ToolUseBlock` — `.name`, `.input`, `.id` — a tool Claude wants to run
- `ThinkingBlock` — `.thinking`, when extended thinking is on
- `ToolResultBlock` — appears in `UserMessage.content`

**Docs:** [Message types](https://code.claude.com/docs/en/agent-sdk/python#message-types)

## 2.5 Give it tools and watch the loop turn

One text turn is not an agent. Let us make a scratch folder with a bug in it and
let Claude actually work.

`cwd` scopes the agent's filesystem access to that directory.

In [ ]:
SCRATCH = ROOT / "workspace" / "_scratch"
SCRATCH.mkdir(parents=True, exist_ok=True)

(SCRATCH / "utils.py").write_text(
    "def calculate_average(numbers):\n"
    "    total = 0\n"
    "    for num in numbers:\n"
    "        total += num\n"
    "    return total / len(numbers)\n"
    "\n"
    "\n"
    "def get_user_name(user):\n"
    "    return user[\"name\"].upper()\n",
    encoding="utf-8",
)
print((SCRATCH / "utils.py").read_text())

Two bugs: `calculate_average([])` divides by zero, and
`get_user_name(None)` raises `TypeError`.

Now the agent. Read the options carefully — they are the whole configuration:

- `allowed_tools` — these run **without prompting**
- `permission_mode="acceptEdits"` — file edits are auto-approved
- `cwd` — the agent only sees this directory
- `max_turns` — a hard stop
- `setting_sources=[]` — do **not** load `CLAUDE.md` or project settings, so
  this notebook behaves identically on every machine

In [ ]:
from claude_agent_sdk import ToolUseBlock

session_id = None

async for message in query(
    prompt="Review utils.py for bugs that would cause crashes. Fix any you find.",
    options=ClaudeAgentOptions(
        allowed_tools=["Read", "Edit", "Glob"],
        permission_mode="acceptEdits",
        cwd=str(SCRATCH),
        max_turns=12,
        setting_sources=[],
        model=MODEL,
    ),
):
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock):
                print(f"  think : {block.text.strip()[:110]}")
            elif isinstance(block, ToolUseBlock):
                print(f"  TOOL  : {block.name} {str(block.input)[:80]}")

    elif isinstance(message, ResultMessage):
        session_id = message.session_id
        print(f"\n  {message.subtype} | {message.num_turns} turns "
              f"| ${message.total_cost_usd or 0:.4f}")

In [ ]:
print((SCRATCH / "utils.py").read_text())

That is the loop from notebook 01, running for real: Claude read the
file, decided what was wrong, edited it, and stopped when it had nothing left to
do. You wrote no dispatch code, no history management, no termination logic.

Note what you *did* write: the tool list, the permission mode, the working
directory, and the turn cap. In the Agent SDK, those four decisions are the
design.

## 2.6 `ResultMessage`: what to check and in what order

`ResultMessage` ends every run. Always check `subtype` **before** reading
`result` — `result` is only populated on success.

| `subtype` | Meaning | `result` present? |
|---|---|---|
| `success` | finished normally | yes |
| `error_max_turns` | hit `max_turns` | no |
| `error_max_budget_usd` | hit `max_budget_usd` | no |
| `error_during_execution` | API failure, cancellation, crash | no |
| `error_max_structured_output_retries` | structured output never validated | no |

Every subtype still carries `total_cost_usd`, `usage`, `num_turns`, and
`session_id`, so you can account for and resume a failed run.

In Python `total_cost_usd`, `usage`, and `model_usage` are typed optional —
check for `None` before reading.

In [ ]:
# Deliberately too few turns, to see an error result.
async def run_with_limit(max_turns):
    try:
        async for message in query(
            prompt="Read utils.py, then write a short README.md describing it.",
            options=ClaudeAgentOptions(
                allowed_tools=["Read", "Write", "Glob"],
                permission_mode="acceptEdits",
                cwd=str(SCRATCH), max_turns=max_turns,
                setting_sources=[], model=MODEL,
            ),
        ):
            if isinstance(message, ResultMessage):
                print(f"  subtype={message.subtype}  turns={message.num_turns}  "
                      f"cost=${message.total_cost_usd or 0:.4f}")
                if message.subtype == "success":
                    print(f"  result: {message.result[:100]}")
    except Exception as exc:
        # IMPORTANT: a single-shot query() RAISES after yielding its error
        # result. The branch above has already run; this catch stops the raise
        # from killing your program.
        print(f"  raised: {type(exc).__name__}: {str(exc)[:80]}")

print("max_turns=1"); await run_with_limit(1)
print("max_turns=12"); await run_with_limit(12)

**Remember this:** `query()` yields the error `ResultMessage` *and then
raises*. Handle both. Your result-handling branch runs first, so wrap the loop in
`try` and treat the exception as "the loop is over", not as "I lost the data".

## 2.7 `query()` vs `ClaudeSDKClient`

Two entry points. Pick by whether you need more than one exchange.

| | `query()` | `ClaudeSDKClient` |
|---|---|---|
| Session | new every call | one session across calls |
| Conversation | single exchange | multi-turn, context retained |
| Interrupts | no | `await client.interrupt()` |
| Change mode mid-run | no | `await client.set_permission_mode(...)` |
| Best for | one-off tasks, batch jobs | chat UIs, interactive apps |

We use `query()` through notebook 05 and pick up `ClaudeSDKClient` in 06.

**Docs:** [Choosing between them](https://code.claude.com/docs/en/agent-sdk/python#choosing-between-query-and-claudesdkclient)

## 2.8 Exercises

1. Change the prompt to `"Add type hints and docstrings to utils.py"`. Count the
   turns. Does it cost more than the bug fix did?
2. Remove `"Edit"` from `allowed_tools` but keep `permission_mode="acceptEdits"`.
   What happens, and why is that not a contradiction?
   *(Hint: `acceptEdits` auto-approves edits at the permission-mode step, so the
   edit still runs. Try `disallowed_tools=["Edit"]` instead and compare.)*
3. Add `"Bash"` to `allowed_tools` and ask it to write tests, run them, and fix
   failures. Watch the verify step of the loop appear.
4. Print `message.usage` on the `ResultMessage`. Which fields are about caching?

In [ ]:
# Scratch space for the exercises.

## Next

[**03 - Configuring the agent**](03_options_tools_permissions.ipynb) — tools,
permission modes, budgets, and system prompts: the levers that turn a demo into
something you would deploy.

---

### References

- [Quickstart](https://code.claude.com/docs/en/agent-sdk/quickstart)
- [Python SDK reference](https://code.claude.com/docs/en/agent-sdk/python)
- [How the agent loop works](https://code.claude.com/docs/en/agent-sdk/agent-loop)
- [Tools reference](https://code.claude.com/docs/en/tools-reference)